# PAYMENTS CLEANING

In [ ]:
import pandas as pd
payments = pd.read_csv("../raw_data/sanlam_raw_payments.csv")

payments.head()

In [ ]:
# Check to see if the transaction date has any NA values 
payments['transaction_date'].isna().sum()


In [ ]:
# Step 1: Create a parser for the known formats
def parse_transaction_date(date_str):
    if pd.isna(date_str):
        return pd.NaT

    date_str = str(date_str).strip()

    try:
        # DD/MM/YYYY
        if '/' in date_str and len(date_str.split('/')[0]) == 2:
            return pd.to_datetime(date_str, format='%d/%m/%Y')

        # MM-DD-YYYY
        elif '-' in date_str and len(date_str.split('-')[0]) == 2:
            return pd.to_datetime(date_str, format='%m-%d-%Y')

        # YYYY-MM-DD
        elif len(date_str.split('-')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y-%m-%d')

        # YYYY/MM/DD
        elif len(date_str.split('/')[0]) == 4:
            return pd.to_datetime(date_str, format='%Y/%m/%d')

        return pd.NaT

    except:
        return pd.NaT

In [ ]:
# Step 2: Parse the dates
payments['parse_transaction_date'] = (
    payments['transaction_date']
    .apply(parse_transaction_date)
)

In [ ]:
payments.head()

In [ ]:
# Step 3: Find malformed dates
bad_dates = payments[
    payments['transaction_date'].notna() &
    payments['parse_transaction_date'].isna()
]

In [ ]:
# Step 4: Count malformed dates
bad_dates.shape[0]

In [ ]:
#Step 5
payments['transaction_date_std'] = (
    payments['parse_transaction_date']
    .dt.strftime('%d-%m-%Y')
)

In [ ]:
payments.columns.tolist()

In [ ]:
payments.head(30)

In [ ]:
#Rename the transaction_date_std to transaction_date_new
payments.rename(columns={'transaction_date_std': 'transaction_date_new'}, inplace=True)

In [ ]:
# Remove the parse_transaction_date coloumn 
payments.drop(columns=['parse_transaction_date'], inplace=True)

In [ ]:
# Remove the transaction_date coloumn (original)
payments.drop(columns=['transaction_date'], inplace=True)

In [ ]:
cols = payments.columns.tolist()

# Remove the column from its current position
cols.remove('transaction_date_new')

# Insert it into the 4th position
cols.insert(3, 'transaction_date_new')

# Reorder the dataframe
payments = payments[cols]

In [ ]:
payments.dtypes

In [ ]:
# Convert string to datetime 
payments['transaction_date_new'] = pd.to_datetime(
    payments['transaction_date_new'],
    format='%d-%m-%Y',
    errors='coerce'
)

In [ ]:
## Check the malformed dates (QUARTINIE ALL 616 ROWS)
payments['transaction_date_new'].isna().sum()

In [ ]:
# Check the currency distinct values 
payments['currency'].unique().tolist()

In [ ]:
payments['currency'] = payments['currency'].replace('R', 'ZAR')

In [ ]:
# Check the distinct values for payment method. 
payments['payment_method'].unique().tolist()

In [ ]:
payments['payment_method'].isna().sum()

In [ ]:
# Set 1202payment_method0 payment method to "Unknown"
payments['payment_method'] = payments['payment_method'].fillna('Unknown')

In [ ]:
payments['payment_method'].head(10)

In [ ]:
payments['payment_method'].unique().tolist()

In [ ]:
#Remove trailling spaces and whitespaces. 
payments['payment_method'] = payments['payment_method'].str.strip()

In [ ]:
payments.head(10)

In [ ]:
# Check if the coloumns contain R or ZAR 
payments[
    payments['amount'].astype(str).str.contains('R|ZAR', na=False)
]

In [ ]:
#Replace the R and ZAR from amounts 
payments['amount'] = payments['amount'].str.replace('ZAR', '', regex=False)
payments['amount'] = payments['amount'].str.replace('R', '', regex=False)
payments['amount'] = payments['amount'].str.strip()

In [ ]:
#Check the distinct values in transcation type
payments['transaction_type'].unique().tolist()

In [ ]:
#Display the value counts for each transaction type
payments['payment_status'].value_counts(dropna=False)

In [ ]:
# Merge success into successful. 
payments['payment_status'] = payments['payment_status'].replace({
    'success': 'Successful'
})

In [ ]:
# Merge paid into Successful
payments['payment_status'] = payments['payment_status'].replace({
    'Paid': 'Successful'
})

In [ ]:
payments['transaction_type'] = payments['transaction_type'].str.strip()

In [ ]:
payments.columns

In [ ]:
#Create a status crosswalk. 
status_crosswalk = {
    'success': 'Successful',
    'Successful': 'Successful',
    'Paid': 'Paid',
    'Reversed': 'Reversed',
    'Failed': 'Failed',
    'Pending': 'Pending'
}

In [ ]:
payments['payment_status'] = (
    payments['payment_status']
    .map(status_crosswalk)
)

In [ ]:
crosswalk = pd.DataFrame({
    'original_value': ['success', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending'],
    'standardized_value': ['Successful', 'Successful', 'Paid', 'Reversed', 'Failed', 'Pending']
})

print(crosswalk)

In [ ]:
#Create a backup if anything goes wrong 
payments_backup = payments.copy()

In [ ]:
# Created a third backup 
payments_v3 = payments_backup.copy() 

In [ ]:
payments.head(10)

In [ ]:
#Counts if the transcation ref appears more than once 
payments['transaction_ref'].value_counts()[payments['transaction_ref'].value_counts() > 1]

In [ ]:
payments.loc[
    payments['policy_id'].astype(str).str.contains('MISSING', na=False)
]

In [ ]:
semi_clean = payments.to_csv("../semi-clean/semi_clean_payments.csv", index = False)

In [ ]:
#payments['policy_id_flag'] = payments['policy_id'].isin(['MISSING', 'MISSING-'])

In [ ]:
payments['A04_flag'] = 'False'
 
payments.loc[
    payments['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'
 

In [ ]:
#Remove trailling spaces and whitespaces from amount
payments['amount'] = payments['amount'].str.strip()

In [ ]:
# Convert object to numeric for amount
payments['amount'] = pd.to_numeric(payments['amount'], errors='coerce') 

In [ ]:
payments.head()

In [ ]:
payments[payments['amount'].isna()]

# PAYMENTS_BACKUP 

In [ ]:
payments_backup.head(10)

In [ ]:
payments_backup['A04_flag'] = 'False'
 
payments_backup.loc[
    payments_backup['policy_id'].str.startswith('MISSING-', na=False),
    'A04_flag'
] = 'True'

In [ ]:
payments_backup.dtypes

In [ ]:
payments_backup['transaction_date_new'].isnull()

In [ ]:
# Create quarantine dataset
quarantine_date = payments_backup[payments_backup['transaction_date_new'].isna()]
 
# Remaining valid records
payments_cleaned = payments_backup[payments_backup['transaction_date_new'].notna()]

In [ ]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_date))
print('Total Records:', len(quarantine_date)+ len(payments_cleaned))
 

In [ ]:
#Creating a new table for the quarantined fsp number
quarantine_date.to_csv("../quarantine/quarantine_date.csv", index=False)

In [ ]:
#Check if all the quaratined date values have moved over to the quarantine file. 
quarantine_date

In [ ]:
payments_cleaned['amount'].unique()

In [ ]:
payments_cleaned['amount'].isnull().sum()

In [ ]:
payments_cleaned.head(10)

In [ ]:
#Create a flag where A16_R1 is displayed. 
payments_cleaned['A16_R1'] = payments_cleaned['amount'].isna()

In [ ]:
# Create quarantine dataset
quarantine_A16_R1 = payments_cleaned[payments_cleaned['A16_R1']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R1']].copy()

In [ ]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R1))
print('Total Records:', len(quarantine_A16_R1)+ len(payments_cleaned))

In [ ]:
#Creating a new table for the quarantined A16_R1 values 
quarantine_A16_R1.to_csv("../quarantine/quarantine_A16_R1.csv", index=False)

In [ ]:
#Create an A16_R3 flag where amount show "not recorded"
payments_cleaned['A16_R3'] = (
    payments_cleaned['amount']
    .str.strip()
    .str.lower()
    .eq('not recorded')
)

In [ ]:
# Create quarantine dataset
quarantine_A16_R3 = payments_cleaned[payments_cleaned['A16_R3']].copy()

# Keep only valid records
payments_cleaned = payments_cleaned[~payments_cleaned['A16_R3']].copy()

In [ ]:
# Checking if the transaction date  is correctly quarantined 
print("Original records:", len(payments_cleaned))
print("Quarantined records:", len(quarantine_A16_R3))
print('Total Records:', len(quarantine_A16_R3)+ len(payments_cleaned))

In [ ]:
#Creating a new table for the quarantined A16_R3 values 
quarantine_A16_R3.to_csv("../quarantine/quarantine_A16_R3.csv", index=False)

In [972]:
payments_cleaned = payments_cleaned.to_csv("../clean_data/payments_cleaned.csv")

# CLAIMS CLEANING

In [ ]:
import pandas as pd
claims = pd.read_csv("../raw_data/sanlam_raw_claims.csv")

claims.head()

In [ ]:
#Cleaned claims_cleaned dataset. 
claims_cleaned = pd.read_csv('../clean_data/claims_clean.csv')

claims_cleaned.head()

In [ ]:
claims.info()

In [ ]:
claims_cleaned.info()

# Checkout

In [ ]:
claims_cleaned.head()

In [ ]:
payments_cleaned.head()